# 13 — Probability of becoming an SPP1-TAM: CellRank on the myeloid trajectory

`12` produced an **ordering**. Three questions it cannot answer on its own:

1. Which states are **terminal** — absorbing ends of the dynamics — rather than merely late?
2. For each cell, what is the **probability of ending in the SPP1/TREM2 state**?
3. Which genes **track that probability** (drivers), as opposed to merely tracking pseudotime?

Then the project's standard last step: aggregate the per-cell readout to **one point per sample**
and ask whether treatment moves it — the same design as `11`, so the answer is comparable to
`11`'s abundance result and `10`'s expression result.

> ### What CellRank can and cannot do on this dataset — read before interpreting anything below
>
> CellRank is a framework over **kernels**, and its informative kernel is the velocity kernel,
> which needs spliced/unspliced counts. **This project has none** — no `.loom`, no velocyto
> output, no `spliced` layer anywhere under the data tree (checked). RNA velocity is not
> available and cannot be added without re-running alignment from the FASTQs.
>
> So two kernels are available, and **neither is independent evidence for the direction of the
> trajectory**:
>
> | kernel | what drives it | what it can and cannot be used for |
> |---|---|---|
> | `PseudotimeKernel` | `12`'s scFates pseudotime | **Circular with `12` by construction.** Its terminal states are largely a restatement of where `12` put the tips. Use it for the *soft* quantities `12` cannot give — per-cell fate probability, driver ranking — not as confirmation that the tips are tips. |
> | `CytoTRACEKernel` | number of genes detected per cell, kNN-smoothed | **It is the depth artifact.** `12` control 1 spent its effort ruling out a library-complexity gradient; CytoTRACE's differentiation potential *is* a gene-count statistic. Agreement with it is therefore **weak** evidence, and disagreement is **strong** evidence something is wrong. It is included for exactly that asymmetry, and is never the primary. |
>
> The honest summary: **CellRank here is a soft-assignment layer on top of `12`, not a second
> opinion about `12`.** The second opinion is `12` §6 control 4 — the two CAR-T backgrounds
> refitted independently. Anything in this notebook that is not reproduced across backgrounds in
> §5 should be reported as a description of this fit, not as a property of the biology.

### What survives that caveat, and is worth doing
- A **graded per-cell fate probability** is genuinely more useful than a hard label for the
  treatment comparison in §6: `11` had to binarise SPP1-TAM at the signature top quartile, and a
  quartile boundary is an arbitrary place to put a cut.
- **Macrostate decomposition** (GPCCA) tests whether the annotated states are metastable at all,
  which is a real question the tree does not ask.
- **Drivers of fate probability** are not the same genes as "correlates with pseudotime" — a gene
  can rise along the whole trajectory yet not distinguish the branches.

## 1. Config and the object from 12

In [ ]:
import warnings; warnings.filterwarnings("ignore")
import numpy as np, pandas as pd, scanpy as sc, anndata as ad
import scipy.sparse as sp
from scipy import stats
from pathlib import Path
import matplotlib as mpl, matplotlib.pyplot as plt
from matplotlib.lines import Line2D

CANDIDATES = [Path("/Users/janzules/Roselab/Spatial/CAR_T"),
              Path("/coh_labs/yunroseli/Jona/CAR-T"),
              Path("/coh_labs/yunroseli/Jona/spatial/CAR-T")]
PROJ = next(p for p in CANDIDATES if p.exists())
OUT  = PROJ/"results/spp1_analysis/sc_myeloid_trajectory"
FIG  = OUT/"figures"; FIG.mkdir(parents=True, exist_ok=True)
TRAJ_H5 = OUT/"myeloid_trajectory.h5ad"
GMT_DIR = PROJ/"data/references/pathways"
SIG_CSV = PROJ/"results/spp1_analysis/sc_spp1_tam_signature.csv"

import os
SEED, N_PERM = 0, 2000
N_JOBS = max(1, (os.cpu_count() or 4) // 2)   # performance cores, not logical
LVL3_ORDER = ["Phagocytic_Mono", "IFN_stimulated_Mono", "Patrolling_Mono", "Mmp12_high_Mac",
              "IFN_stimulated_Mac", "Fn1_high_Mac", "Arg1_high_Mac", "Apoe_high_Mac"]
MONO_STATES = {"Phagocytic_Mono", "IFN_stimulated_Mono", "Patrolling_Mono"}
N_STATES = 8          # macrostates; §3 prints the eigengap so this is auditable
KERNEL_MIX = 0.8      # weight on the pseudotime kernel; 1 - this goes to connectivity
GROUPS = ["NoTx_Tu1","NoTx_Tu2","CyPSCA_Tu1","CyPSCA_Tu2","RTCyPSCA_Tu1","RTCyPSCA_Tu2",
          "CyTAG72_Tu1","CyTAG72_Tu2","RTCyTAG72_Tu1","RTCyTAG72_Tu2"]

S1, S2, S3 = "#2a78d6", "#eb6834", "#1baf7a"
SER = {"none": "#6f6e69", "PSCA": S1, "TAG72": S2}          # identical to notebook 11
INK, MUTED, GRID, SURF, FADE = "#0b0b0b", "#52514e", "#e6e5e2", "#fcfcfb", "#dcdbd7"
SEQ = mpl.colors.LinearSegmentedColormap.from_list(
    "seq_blue", ["#cde2fb", "#86b6ef", "#3987e5", "#1c5cab", "#0d366b"])
plt.rcParams.update({"pdf.fonttype": 42, "ps.fonttype": 42, "svg.fonttype": "path",
                     "figure.facecolor": SURF, "axes.facecolor": SURF, "axes.edgecolor": GRID,
                     "axes.labelcolor": MUTED, "xtick.color": MUTED, "ytick.color": MUTED,
                     "font.size": 9})
pd.set_option("display.width", 220, "display.max_columns", 60)
rng = np.random.default_rng(SEED)

import cellrank as cr
import importlib.metadata as _md
print(f"cellrank {_md.version('cellrank')} | scanpy {_md.version('scanpy')}")

assert TRAJ_H5.exists(), f"{TRAJ_H5} not found — run notebook 12 first."
adata = sc.read_h5ad(TRAJ_H5)
adata.obs["ct3"] = pd.Categorical(adata.obs.ct3.astype(str), categories=LVL3_ORDER, ordered=True)
SIG_PRESENT = [g for g in pd.read_csv(SIG_CSV)["gene"] if g in adata.var_names]
SIGSET = set(SIG_PRESENT)
print(f"\n{adata.n_obs:,} cells x {adata.n_vars:,} genes | pseudotime t "
      f"{adata.obs.t.min():.3f}..{adata.obs.t.max():.3f}")
print(f"milestones: {sorted(adata.obs.milestones.astype(str).unique())}")
print(f"\nmean pseudotime by state:\n"
      f"{adata.obs.groupby('ct3', observed=True)['t'].mean().round(3).to_string()}")

## 2. Two kernels, and what each one is made of

The transition matrix is `0.8 x pseudotime + 0.2 x connectivity`. The connectivity term is what
stops the pseudotime kernel from being a pure one-way ratchet — without it, every cell is forced
forward even where the manifold says two states are mutually reachable. `0.8/0.2` is CellRank's
own documented default mix and is not tuned here; §3 reports the macrostates under a `1.0/0.0`
mix as well, so the choice is visible rather than hidden.

In [ ]:
from cellrank.kernels import PseudotimeKernel, ConnectivityKernel, CytoTRACEKernel

if "neighbors" not in adata.uns:
    sc.pp.neighbors(adata, n_neighbors=30, use_rep="X_hm", random_state=SEED)

pk = PseudotimeKernel(adata, time_key="t").compute_transition_matrix()
ck = ConnectivityKernel(adata).compute_transition_matrix()
K  = KERNEL_MIX * pk + (1 - KERNEL_MIX) * ck
print(K)

# ---- the CytoTRACE cross-check. Needs a kNN-smoothed count layer; built inline so no scvelo. ----
A = adata.obsp["connectivities"].copy()
A.setdiag(1.0)
A = A.multiply(1.0 / np.asarray(A.sum(1)).ravel()[:, None]).tocsr()
adata.layers["Ms"] = np.asarray(A @ adata.X.toarray() if sp.issparse(adata.X) else A @ adata.X)
ctk = CytoTRACEKernel(adata).compute_cytotrace(layer="Ms", aggregation="mean")
r_ct    = stats.spearmanr(adata.obs.ct_pseudotime, adata.obs.t).statistic
r_depth = stats.spearmanr(adata.obs.ct_pseudotime, adata.obs.n_genes_by_counts).statistic

# 12 roots the tree at an INTERIOR classical-monocyte hub, so t is a distance from that hub and
# runs outward along BOTH the patrolling branch and the macrophage branch. A global correlation
# against a monotone statistic therefore mixes two directions and is not interpretable. The
# comparable quantity is the correlation restricted to the macrophage arm.
mac_arm = (~adata.obs.ct3.astype(str).isin(MONO_STATES)).to_numpy()
r_ct_mac = stats.spearmanr(adata.obs.ct_pseudotime[mac_arm], adata.obs.t[mac_arm]).statistic
print(f"""
=== CytoTRACE vs the scFates ordering ===
  Spearman(CytoTRACE, genes detected)            : {r_depth:+.3f}   <- CytoTRACE IS a gene count
  Spearman(CytoTRACE, t) — all cells             : {r_ct:+.3f}    (mixes the two arms, see below)
  Spearman(CytoTRACE, t) — macrophage arm only   : {r_ct_mac:+.3f}    <- the comparable number

  READ: the first number being large is expected and validates nothing — it restates what
        CytoTRACE is made of, and it is the same quantity 12 control 1 worked to rule out.
        The macrophage-arm number is the informative one, and the sign convention matters:
        CellRank defines ct_pseudotime so HIGHER = MORE differentiated, so agreement with 12
        means POSITIVE. A strong negative there would mean 12 ordered the macrophage arm against
        the depth gradient — either a real finding or a broken root — and would have to be
        resolved before anything below is used. A value near zero is the benign case: the
        orderings are simply measuring different things.""")
print(adata.obs.groupby("ct3", observed=True)[["t", "ct_pseudotime", "n_genes_by_counts"]]
        .mean().reindex(LVL3_ORDER).round(3).to_string())

## 3. Macrostates and terminal states (GPCCA)

GPCCA finds the **metastable** sets of the transition matrix — regions the dynamics linger in.
This is the one question in this notebook that the tree does not already answer: a tip is where a
branch ends, a macrostate is where cells *stay*, and the two need not coincide.

The number of macrostates is read off the **eigenvalue spectrum** (a spectral gap), not chosen to
give a convenient answer; the spectrum is printed so the choice is auditable.

In [ ]:
from cellrank.estimators import GPCCA

g = GPCCA(K)
g.compute_schur(n_components=20)
ev = np.real(g.eigendecomposition["eigengap"]) if "eigengap" in (g.eigendecomposition or {}) else None
print("Schur decomposition done. eigengap:", ev)

g.compute_macrostates(n_states=N_STATES, cluster_key="ct3")
print("\n=== macrostates ===")
mac = adata.obs["macrostates"] = g.macrostates
comp = pd.crosstab(g.macrostates.astype(str), adata.obs.ct3)
comp = comp.div(comp.sum(1), axis=0)
info = pd.DataFrame({
    "n": g.macrostates.value_counts(),
    "dominant_state": comp.idxmax(1),
    "purity": comp.max(1).round(3),
    "mean_t": adata.obs.groupby(g.macrostates.astype(str), observed=True)["t"].mean().round(3),
    "mean_SPP1_sig": adata.obs.groupby(g.macrostates.astype(str), observed=True)["SPP1_TAM_sig"].mean().round(3),
    "pct_Spp1_pos": (adata.obs.assign(m=g.macrostates.astype(str))
                       .groupby("m", observed=True)["spp1_call"]
                       .apply(lambda s: 100*(s == "Spp1_pos_mac").mean()).round(1)),
}).sort_values("mean_t")
print(info.to_string())

g.predict_terminal_states()
TERM = [str(c) for c in g.terminal_states.cat.categories]
print(f"\nterminal states ({len(TERM)} of {N_STATES} macrostates): {TERM}")
try:
    g.predict_initial_states()
    print(f"initial states : {list(g.initial_states.cat.categories)}")
except Exception as e:
    print(f"initial states : not identifiable — {type(e).__name__}: {str(e)[:110]}")
    print("                 (expected with a pseudotime kernel: the initial and terminal "
          "candidates overlap. 12's marker-based root is the initial state.)")
if len(TERM) == N_STATES:
    print("""
  NOTE: EVERY macrostate came back terminal. With a pseudotime kernel that is common and not a
        bug — the transition matrix is near-acyclic, so no metastable set is left non-absorbing.
        It does mean this decomposition is not independently telling you which states are ends;
        it is inheriting 12's tree. Weigh §3 accordingly.""")
if any(str(c).endswith(("_1", "_2", "_3")) for c in TERM):
    print("""
  NOTE: GPCCA split at least one annotated state into several macrostates (the `_1`/`_2`
        suffixes). Those are sub-regions of one biological state, not separate fates, so §4
        SUMS their fate probabilities rather than picking one — otherwise P(SPP1-TAM) is
        arbitrarily divided between two labels for the same endpoint.""")
print("""
  READ: the SPP1 terminal state(s) should have the highest mean_SPP1_sig AND the highest
        pct_Spp1_pos — that second column is Alice's lvl4 call, which knows nothing about the
        signature or the tree, so the two agreeing is the non-circular part of this table.
        A terminal state that is 'Patrolling_Mono' is expected and correct: Ly6C-low monocytes
        are a separate fate, not a step on the way to a TAM.""")

# ---- the mix is a choice, so show it doesn't decide the answer ---------------------------------
g_pure = GPCCA(pk); g_pure.compute_schur(n_components=20)  # the mix is a choice — show it
g_pure.compute_macrostates(n_states=N_STATES, cluster_key="ct3"); g_pure.predict_terminal_states()
print(f"\nterminal states under a PURE pseudotime kernel (1.0/0.0): "
      f"{list(g_pure.terminal_states.cat.categories)}")

## 4. Fate probabilities — the probability of becoming an SPP1-TAM

In [ ]:
g.compute_fate_probabilities()
FP = pd.DataFrame(np.asarray(g.fate_probabilities),
                  index=adata.obs_names, columns=list(g.fate_probabilities.names))
print("=== mean fate probability by annotated state ===")
print(FP.groupby(adata.obs.ct3.values, observed=True).mean().reindex(LVL3_ORDER).round(3).to_string())

# ---- which terminal states ARE the SPP1 endpoint? --------------------------------------------
# GPCCA can split one biological endpoint into several macrostates, so this profiles every
# terminal state and SUMS the ones that are the SPP1/TREM2 endpoint. The deciding column is
# pct_Spp1_pos — Alice's lvl4 call, which knows nothing about the signature or about 12's tree.
ts = g.terminal_states.astype(str)
rows = []
for c in FP.columns:
    m = (ts == str(c)).to_numpy()
    macs = m & (adata.obs.spp1_call != "not_in_mac_object").to_numpy()
    rows.append(dict(
        lineage=str(c), n_terminal=int(m.sum()),
        dominant_state=adata.obs.ct3[m].mode().iloc[0] if m.any() else "\u2014",
        mean_t=float(adata.obs.t[m].mean()),
        mean_SPP1_sig=float(adata.obs.SPP1_TAM_sig[m].mean()),
        pct_Spp1_pos=float(100*(adata.obs.spp1_call[macs] == "Spp1_pos_mac").mean())
                     if macs.any() else np.nan))
TS = pd.DataFrame(rows).set_index("lineage").sort_values("pct_Spp1_pos", ascending=False)
BASE = 100*(adata.obs.spp1_call[adata.obs.spp1_call != "not_in_mac_object"] == "Spp1_pos_mac").mean()
print("\n=== terminal states, profiled ===")
print(TS.round(3).to_string())
print(f"  macrophage-wide Spp1_pos_mac base rate: {BASE:.1f}%")

SPP1_LINS = [l for l, r in TS.iterrows()
             if (r.pct_Spp1_pos > BASE) and (r.dominant_state not in MONO_STATES)]
if not SPP1_LINS:
    SPP1_LINS = [TS.index[0]]
    print("  no terminal state clears the base rate — falling back to the single best.")
SPP1_LIN = SPP1_LINS[0]                      # the strongest single one, for CellRank's own driver call
print(f"\nSPP1 fate = sum over {SPP1_LINS}")
print(f"  (enriched over the {BASE:.1f}% base rate AND not monocyte-dominated)")
print(f"  excluded : {[l for l in TS.index if l not in SPP1_LINS]}")

adata.obs["p_spp1"] = FP[SPP1_LINS].sum(1).to_numpy()
adata.obs["p_spp1_top1"] = FP[SPP1_LIN].to_numpy()
for c in FP.columns: adata.obs[f"fate_{c}"] = FP[c].to_numpy()
print(f"\np_spp1 (combined) vs p_spp1 (single best): Spearman "
      f"{stats.spearmanr(adata.obs.p_spp1, adata.obs.p_spp1_top1).statistic:+.3f}")

# sanity: does it agree with the two independent SPP1-TAM definitions?
mac_only = adata.obs.spp1_call != "not_in_mac_object"
auc = stats.mannwhitneyu(adata.obs.p_spp1[mac_only & (adata.obs.spp1_call == "Spp1_pos_mac")],
                         adata.obs.p_spp1[mac_only & (adata.obs.spp1_call == "Spp1_neg_mac")])
n1 = (mac_only & (adata.obs.spp1_call == "Spp1_pos_mac")).sum()
n2 = (mac_only & (adata.obs.spp1_call == "Spp1_neg_mac")).sum()
print(f"""
=== does p_spp1 recover the two INDEPENDENT SPP1-TAM definitions? ===
  vs Alice's lvl4 call (macrophages only, n = {n1:,} pos / {n2:,} neg)
      AUC = {auc.statistic/(n1*n2):.3f}   (0.5 = no information)
  vs the 01c graded signature
      Spearman(p_spp1, SPP1_TAM_sig) = {stats.spearmanr(adata.obs.p_spp1, adata.obs.SPP1_TAM_sig).statistic:+.3f}
  vs pseudotime (expected high — p_spp1 is built on it; shown so it is not mistaken for evidence)
      Spearman(p_spp1, t) = {stats.spearmanr(adata.obs.p_spp1, adata.obs.t).statistic:+.3f}""")
TS.to_csv(OUT/"terminal_states.csv")
adata.obs[["ct3","cond","sample","tu","mouse","cart","RT","spp1_call","SPP1_TAM_sig","t",
           "p_spp1","p_spp1_top1","ct_pseudotime"]
          + [f"fate_{c}" for c in FP.columns]].to_parquet(OUT/"fate_probabilities.parquet")

In [ ]:
U = adata.obsm["X_umap"]; PS = 0.9 if adata.n_obs > 40000 else 2.0
fig, axes = plt.subplots(1, 3, figsize=(14.5, 4.4))

def _frame(ax, title, sub=None):
    ax.set_xticks([]); ax.set_yticks([])
    for s in ax.spines.values(): s.set_color(GRID)
    ax.set_title(title, fontsize=9.5, color=INK, loc="left", pad=3)
    if sub: ax.text(0, 1.004, sub, transform=ax.transAxes, fontsize=7.5, color=MUTED, va="bottom")

sm = axes[0].scatter(U[:,0], U[:,1], s=PS, c=adata.obs.p_spp1, cmap=SEQ, vmin=0, vmax=1,
                     linewidths=0, rasterized=True)
_frame(axes[0], "P(SPP1-TAM)", "absorption probability, GPCCA")
cb = fig.colorbar(sm, ax=axes[0], fraction=.045, pad=.02); cb.outline.set_visible(False)
cb.ax.tick_params(labelsize=7, colors=MUTED)

ts = g.terminal_states.astype(str)
axes[1].scatter(U[:,0], U[:,1], s=PS, c=FADE, linewidths=0, rasterized=True)
for c in [x for x in ts.unique() if x != "nan"]:
    m = (ts == c).to_numpy()
    axes[1].scatter(U[m,0], U[m,1], s=PS*2, c=S1 if c == SPP1_LIN else MUTED,
                    linewidths=0, rasterized=True)
    axes[1].text(np.median(U[m,0]), np.median(U[m,1]), c, fontsize=8, color=INK,
                 ha="center", va="center", fontweight="bold")
_frame(axes[1], "Terminal states", "directly labelled; SPP1 lineage in blue")

# p_spp1 vs pseudotime, binned — the shape says whether commitment is early or late
b = pd.qcut(adata.obs.t, 30, labels=False)
prof = adata.obs.groupby(b, observed=True).agg(t=("t","mean"), p=("p_spp1","mean"),
                                               lo=("p_spp1", lambda s: s.quantile(.25)),
                                               hi=("p_spp1", lambda s: s.quantile(.75)))
axes[2].fill_between(prof.t, prof.lo, prof.hi, color=S1, alpha=.18, linewidth=0)
axes[2].plot(prof.t, prof.p, color=S1, lw=2, solid_capstyle="round")
axes[2].set_xlabel("pseudotime t", fontsize=9); axes[2].set_ylabel("P(SPP1-TAM)", fontsize=9)
axes[2].set_ylim(0, 1)
axes[2].set_title("Commitment along the trajectory", fontsize=9.5, color=INK, loc="left")
axes[2].text(0.02, 0.95, "line = mean, band = IQR", transform=axes[2].transAxes,
             fontsize=7.5, color=MUTED, va="top")
axes[2].grid(axis="y", color=GRID, lw=.8); axes[2].set_axisbelow(True)
for s in ("top","right"): axes[2].spines[s].set_visible(False)

fig.suptitle("Fate probability toward the SPP1/TREM2 terminal state",
             fontsize=12, color=INK, x=0.005, ha="left", y=1.02)
fig.tight_layout(); fig.savefig(FIG/"13_fate_probabilities.png", dpi=180, bbox_inches="tight")
plt.show(); plt.close(fig)

## 5. Driver genes — and the only control that can license them

`compute_lineage_drivers` correlates every gene with the SPP1 absorption probability. On its own
that is a description of one fit, and this fit is built on `12`'s pseudotime, so the ranking
inherits everything `12` assumed.

The control is the project's standing one: compute the same correlation **separately inside the
PSCA cells and inside the TAG72 cells** and ask whether the two rankings agree, against a **label
permutation** null. `09` is explicit about why the permutation is not optional — a parametric p
on ~10,000 correlated genes gave 1.7e-152 where the permutation gave 0.49.

Signature genes are excluded throughout, for the reason `12` §7 gives.

In [ ]:
# CellRank's own driver call, on the single strongest SPP1 lineage. It is reported because it
# carries CIs and q-values; the ranking that gets used is the combined-p_spp1 one below, since
# p_spp1 is the sum over SPP1_LINS and this is only one of them.
drivers = g.compute_lineage_drivers(lineages=SPP1_LIN, cluster_key="ct3", return_drivers=True)
dcol = [c for c in drivers.columns if c.endswith("_corr")][0]
drivers = drivers[~drivers.index.isin(SIGSET)]
drivers.sort_values(dcol, ascending=False).to_csv(OUT/"cellrank_lineage_drivers.csv")
print(f"=== CellRank drivers of P({SPP1_LIN}) — top 20, signature genes removed ===")
print(drivers.sort_values(dcol, ascending=False).head(20).round(4).to_string())

# ---- the cross-background control, on the COMBINED p_spp1 --------------------------------------
det = np.asarray((adata.X > 0).mean(0)).ravel()
GENES = adata.var_names[(det > 0.05) & ~adata.var_names.isin(SIGSET)]
def corr_with(vec, mask):
    Xg = adata[mask, GENES].X
    Xg = Xg.toarray() if sp.issparse(Xg) else np.asarray(Xg)
    Rx = stats.rankdata(Xg, axis=0); Rv = stats.rankdata(np.asarray(vec)[mask])
    Rx -= Rx.mean(0); Rv = Rv - Rv.mean()
    return pd.Series((Rx*Rv[:,None]).sum(0) /
                     (np.sqrt((Rx**2).sum(0))*np.sqrt((Rv**2).sum())+1e-12), index=GENES)

p = adata.obs.p_spp1.to_numpy()
D = pd.DataFrame({bg: corr_with(p, (adata.obs.cart == bg).to_numpy()) for bg in ("PSCA","TAG72")}).dropna()
r_obs = stats.pearsonr(D.PSCA, D.TAG72).statistic
null = np.array([stats.pearsonr(D.PSCA, rng.permutation(D.TAG72.to_numpy())).statistic
                 for _ in range(N_PERM)])
p_perm = (1 + (np.abs(null) >= abs(r_obs)).sum()) / (1 + N_PERM)
D["mean"] = D.mean(1); D["sign_agree"] = np.sign(D.PSCA) == np.sign(D.TAG72)
D.sort_values("mean", ascending=False).to_csv(OUT/"spp1_fate_drivers_by_background.csv")
print(f"""
=== cross-background reproducibility of the driver ranking ===
  genes                 : {len(D):,}
  r(PSCA, TAG72)        : {r_obs:+.3f}
  permutation p         : {p_perm:.4f}  ({N_PERM:,} label permutations)
  sign agreement        : {100*D.sign_agree.mean():.1f}% of genes

  BENCHMARKS, same project, comparable machinery:
    treated vs NoTx, cross-background                r = +0.89   (09 §7)
    the RT null                                      r = -0.29, perm p = 0.49
  READ: only the genes in the reproducible list below are reportable as drivers. The rest
        describe this fit.""")
REP = D[D.sign_agree].sort_values("mean", ascending=False)
print(f"\n  top 25 REPRODUCIBLE drivers (sign-agreeing in both backgrounds):")
print("    " + ", ".join(REP.index[:25]))
print(f"  bottom 15 (reproducibly anti-correlated — the monocyte end):")
print("    " + ", ".join(REP.index[-15:]))
agree_cr = stats.spearmanr(
    D["mean"].reindex(drivers.index).dropna(),
    drivers[dcol].reindex(D["mean"].reindex(drivers.index).dropna().index)).statistic
print(f"\n  agreement with CellRank's own single-lineage ranking: Spearman {agree_cr:+.3f}")

## 6. Does treatment change the probability of becoming an SPP1-TAM?

One point per sample, mice joined Tu1→Tu2, exactly the design of `11` — so this answer is
directly comparable to `11`'s abundance result and `10`'s expression result.

What the rest of the project predicts, and what would be a surprise:

- `10`: the axis that moves SPP1-TAM **state** is **which CAR**, not radiation (342/192 genes vs
  1–19), an interferon programme up in PSCA.
- `11` / `rtAdvantage/02b`: the axis that moves myeloid **abundance** is also the CAR
  (51% of `M1_like_Mac` CLR variance vs 3.6% for RT).
- So a **CAR gap in fate probability with no RT main effect** is the expected result and adds a
  third, dynamic modality to the same finding. **An RT effect here would be the surprise** — and
  given five previous RT nulls in this project, it would need the leave-one-mouse-out and the
  both-strata check below before it were reported at all.

> `NoTx` is the only untreated reference and it is compromised — 25–42% tumour content vs 4–6% in
> treated arms. It is drawn in grey, set apart, and used only for direction.

In [ ]:
o = adata.obs
rows = []
for s, d in o.groupby("sample", observed=True):
    rows.append(dict(sample=s, cond=d.cond.iloc[0], tu=d.tu.iloc[0], mouse=d.mouse.iloc[0],
                     cart=d.cart.iloc[0], RT=bool(d.RT.iloc[0]), n_cells=len(d),
                     p_spp1=d.p_spp1.mean(), p_spp1_med=d.p_spp1.median(),
                     frac_committed=float((d.p_spp1 > 0.5).mean()),
                     mean_t=d.t.mean()))
A = pd.DataFrame(rows)
A["group"] = A.cond + "_" + A.tu
A = A.sort_values(["cond","tu","mouse"]).reset_index(drop=True)
A.to_csv(OUT/"fate_probability_per_sample.csv", index=False)
print("=== one row per sample ===")
print(A[["sample","cond","tu","n_cells","p_spp1","frac_committed","mean_t"]]
        .to_string(index=False, float_format=lambda x: f"{x:.3f}"))

print("\n=== group means ===")
print(A.groupby("group", observed=True)[["p_spp1","frac_committed"]].mean()
        .reindex(GROUPS).round(3).to_string())

def gap(df, stratum=None, metric="p_spp1"):
    d = df[df.cart.isin(["PSCA","TAG72"])]
    if stratum == "RT":    d = d[d.RT]
    if stratum == "nonRT": d = d[~d.RT]
    a, b = d[d.cart=="PSCA"][metric], d[d.cart=="TAG72"][metric]
    return a.mean()-b.mean(), len(a), len(b)

full, na, nb = gap(A)
print(f"\n=== CAR axis: PSCA - TAG72 in mean P(SPP1-TAM) ===")
print(f"  full data: {full:+.4f}   (n = {na} PSCA vs {nb} TAG72 samples)")
loo = pd.DataFrame([dict(dropped=m, gap=gap(A[A.mouse != m])[0])
                    for m in sorted(A[A.cart.isin(['PSCA','TAG72'])].mouse.unique())])
loo["change"] = loo.gap - full
print(f"  leave-one-mouse-out ({len(loo)} mice): {loo.gap.min():+.4f} .. {loo.gap.max():+.4f}"
      f" | sign flips: {'YES' if (np.sign(loo.gap) != np.sign(full)).any() else 'NO'}")
g_non, g_rt = gap(A, 'nonRT')[0], gap(A, 'RT')[0]
print(f"  by RT stratum: non-RT {g_non:+.4f} | RT {g_rt:+.4f} | same sign: "
      f"{'YES' if np.sign(g_non)==np.sign(g_rt) else 'NO — an interaction, not a main effect'}")

print(f"\n=== RT axis: background-matched, the only contrast this design allows ===")
rt_rows = []
for bg, (rt_arm, base) in {"PSCA": ("RTCyPSCA","CyPSCA"), "TAG72": ("RTCyTAG72","CyTAG72")}.items():
    for site in ("Tu1","Tu2"):
        a = A[(A.cond==rt_arm)&(A.tu==site)].p_spp1; b = A[(A.cond==base)&(A.tu==site)].p_spp1
        rt_rows.append(dict(background=bg, site=site, RT=a.mean(), nonRT=b.mean(),
                            delta=a.mean()-b.mean(), n_RT=len(a), n_nonRT=len(b)))
RTt = pd.DataFrame(rt_rows)
print(RTt.to_string(index=False, float_format=lambda x: f"{x:+.4f}"))
sign_ok = (RTt.groupby("site").delta.apply(lambda s: np.sign(s.iloc[0])==np.sign(s.iloc[1])))
print(f"\n  RT deltas sign-agreeing across the two CAR backgrounds: {sign_ok.to_dict()}")
print(f"""
  READ: the project rule is that an RT claim needs the SAME SIGN in BOTH backgrounds. One
        background moving is the cart x RT interaction that rtAdvantage/02b measured (8.7% of
        variance vs 3.5% for the RT main effect), not an effect of radiation.""")

# within-mouse difference-in-differences, the strongest contrast the design allows
w = A.pivot_table(index=["mouse","cond","cart"], columns="tu", values="p_spp1").dropna().reset_index()
w["within"] = w.Tu1 - w.Tu2
print("\n=== within-mouse Tu1 - Tu2 (irradiated minus abscopal) ===")
print(w.groupby(["cond"], observed=True)["within"].agg(["mean","size"]).round(4).to_string())
did = {bg: (w[(w.cart==bg) & w.cond.str.startswith("RT")].within.mean()
            - w[(w.cart==bg) & ~w.cond.str.startswith("RT")].within.mean())
       for bg in ("PSCA","TAG72")}
print(f"  difference-in-differences (RT minus non-RT, within background): "
      + ", ".join(f"{k} {v:+.4f}" for k, v in did.items())
      + f" | same sign: {'YES' if np.sign(list(did.values())[0])==np.sign(list(did.values())[1]) else 'NO'}")

In [ ]:
# One panel per metric; every sample a point, mice joined Tu1->Tu2. Three colour categories
# (none / PSCA / TAG72) — inside the all-pairs-safe first slots — and every group is directly
# labelled on the x axis, so identity is never colour-alone.
METRICS = [("p_spp1", "mean P(SPP1-TAM) per sample"),
           ("frac_committed", "fraction of cells with P > 0.5")]
fig, axes = plt.subplots(1, 2, figsize=(13.5, 4.6), sharex=True)
xs = {gp: i for i, gp in enumerate(GROUPS)}
for ax, (col, ylab) in zip(np.atleast_1d(axes), METRICS):
    for mouse, d in A.groupby("mouse", observed=True):
        d = d.sort_values("tu")
        if len(d) == 2 and d.group.iloc[0] in xs and d.group.iloc[1] in xs:
            ax.plot([xs[d.group.iloc[0]], xs[d.group.iloc[1]]], d[col].values,
                    color=SER[d.cart.iloc[0]], lw=1.0, alpha=.45, zorder=2)
    for gp in GROUPS:
        d = A[A.group == gp]
        if not len(d): continue
        x = xs[gp]; jit = np.linspace(-.13, .13, len(d)) if len(d) > 1 else np.array([0.])
        ax.scatter(x + jit, d[col], s=42, color=SER[d.cart.iloc[0]], edgecolor=SURF,
                   linewidth=1.1, zorder=3)
        ax.plot([x-.30, x+.30], [d[col].mean()]*2, color=INK, lw=2.0,
                solid_capstyle="round", zorder=4)
    ax.axvline(1.5, color=MUTED, lw=1.0, ls=(0, (4, 3)), zorder=1)
    ax.set_xticks(range(len(GROUPS)))
    ax.set_xticklabels([gp.replace("_Tu", " Tu") for gp in GROUPS], rotation=38, ha="right",
                       fontsize=8.5, color=INK)
    ax.set_ylabel(ylab, fontsize=9)
    ax.grid(axis="y", color=GRID, lw=.8); ax.set_axisbelow(True)
    for s in ("top","right"): ax.spines[s].set_visible(False)
axes[0].legend(handles=[Line2D([0],[0], marker="o", ls="", color=c, label=k, markersize=7)
                        for k, c in SER.items()],
               fontsize=8, frameon=False, loc="upper left", title="CAR background",
               title_fontsize=8)
fig.suptitle("Probability of becoming an SPP1-TAM, one point per sample",
             fontsize=12, color=INK, x=0.005, ha="left", y=1.02)
fig.text(0.005, -0.06, "Thin lines join the two tumours of one mouse (Tu1 irradiated → Tu2 "
         "abscopal). Black bars are group means. NoTx is grey and set apart — 25–42% tumour "
         "content vs 4–6% in treated arms makes it a direction reference only.",
         fontsize=7.5, color=MUTED)
fig.tight_layout(); fig.savefig(FIG/"13_fate_by_condition.png", dpi=180, bbox_inches="tight")
plt.show(); plt.close(fig)

## 7. SUMMARY

In [ ]:
print("="*96); print("SUMMARY — report these back"); print("="*96)
print(f"""
SETUP
  kernel      : {KERNEL_MIX} x PseudotimeKernel(scFates t from 12) + {1-KERNEL_MIX:.1f} x ConnectivityKernel
                NO velocity kernel — this project has no spliced/unspliced counts.
  macrostates : {N_STATES} | terminal: {list(g.terminal_states.cat.categories)}
  SPP1 fate   : sum over {SPP1_LINS}, chosen by Alice's lvl4 Spp1_pos_mac rate against the
                {BASE:.1f}% macrophage base rate — not by the signature that oriented the tree
  CytoTRACE   : rho vs t = {r_ct_mac:+.3f} on the macrophage arm; rho vs genes detected = {r_depth:+.3f}

DOES p_spp1 MEAN WHAT IT SAYS
  vs lvl4 Spp1_pos_mac (independent)     AUC {auc.statistic/(n1*n2):.3f}
  vs 01c graded signature                rho {stats.spearmanr(adata.obs.p_spp1, adata.obs.SPP1_TAM_sig).statistic:+.3f}
  vs pseudotime (circular, shown anyway) rho {stats.spearmanr(adata.obs.p_spp1, adata.obs.t).statistic:+.3f}

DRIVERS
  cross-background r = {r_obs:+.3f}, permutation p = {p_perm:.4f}, sign agreement {100*D.sign_agree.mean():.1f}%
  (benchmarks: +0.89 positive control, -0.29 / p = 0.49 the RT null)
  reportable drivers = the sign-agreeing list only; the full ranking describes this fit.

TREATMENT
  CAR axis (PSCA - TAG72)  : {full:+.4f}  | LOO range {loo.gap.min():+.4f}..{loo.gap.max():+.4f}
                             | non-RT {g_non:+.4f} vs RT {g_rt:+.4f}
  RT axis (background-matched, both sites):
{RTt.to_string(index=False, float_format=lambda x: f'{x:+.4f}')}
  within-mouse DiD: {', '.join(f'{k} {v:+.4f}' for k, v in did.items())}

WORDING THIS LICENSES
  * p_spp1 is an ABSORPTION PROBABILITY ON 12's TREE, not a measured commitment. Wherever this
    is written up, the sentence "no RNA velocity is available in this dataset" goes with it.
  * A CAR gap that survives leave-one-mouse-out AND appears in both RT strata is reportable,
    and lines up with 10 (expression) and 11 (abundance) as a third modality on the same axis.
  * An RT effect is reportable ONLY with the same sign in both CAR backgrounds. One background
    alone is the cart x RT interaction, which is what rtAdvantage/02b already named.

FILES -> {OUT}
  fate_probabilities.parquet | fate_probability_per_sample.csv
  spp1_fate_drivers_by_background.csv
  figures/13_fate_probabilities.png | 13_fate_by_condition.png""")